# 📋 [K14] 군 생활 데이터 분석 - 생활관 생활 데이터로 개선 과제 찾기
### [정답 해설지] 모범 프롬프트 & 산출물 설계서 (Solution Guide)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K14/K14_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 병영생활·부대관리 (Level 1) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 15.0시간) |
| **실습 유형** | 군 생활 데이터 통계 가공, HITL 수기 검산, 시각화 왜곡 방지 및 종합 보고서 |

### 📖 프로젝트 목표
생활관은 체력단련실·사이버지식정보방·PX 이용, 수면, 식단, 건의 같은 생활 데이터가 쌓이지만, 숫자를 표로만 두면 어디가 나아졌고 어디가 나빠지는지 알기 어렵습니다. 이 프로젝트에서는 가상 한빛대대 3개 중대의 4주간 생활관 생활 데이터를 생성형 AI로 요약하고 추세와 중대별 차이를 찾고, 그래프를 요청하고, 원인 가설과 개선 제안을 만든 뒤 발표 자료까지 정리합니다. AI가 계산한 수치를 직접 확인하는 습관과, 데이터만으로 원인을 단정하지 않는 태도를 함께 익힙니다.

> **⚠️ [교육용 가상 자료 및 보안 수칙 준수 원칙]**
> - 본 워크북의 모든 데이터는 장병 복지 증진을 위한 **교육용 가상 부대 데이터셋(가상 한빛대대)**입니다.
> - 실제 부대 현황, 장병 인적사항(성명, 군번 등) 및 비인가 군사자료는 절대 외부 AI 서비스에 입력하지 않습니다.
> - AI가 산출한 수치는 반드시 원문과 대조하여 검산(HITL)하며 최종 부대 개선 정책 결정은 지휘계통에 따릅니다.



In [ ]:
# ============================================================================
# [Colab 환경 설정 및 헬퍼 함수] 데이터 분석, 가상 데이터 로더 & 마크다운 렌더러
# ============================================================================
!pip install -q pandas numpy tabulate matplotlib seaborn

import pandas as pd
import numpy as np
import json
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# 폰트 설정 (Colab / Linux 호환)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# ----------------------------------------------------------------------------
# [데이터셋 자동 로드] 가상 한빛대대 4주간 생활관 생활 데이터 (12행)
# ----------------------------------------------------------------------------
data_records = [
    {'주차': '1주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 112, '싸지방': 54, 'PX': 140, '수면만족': 3.8, '식단만족': 3.9, '건의': 3},
    {'주차': '1주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 98, '싸지방': 31, 'PX': 160, '수면만족': 3.6, '식단만족': 3.5, '건의': 4},
    {'주차': '1주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 104, '싸지방': 47, 'PX': 118, '수면만족': 3.7, '식단만족': 3.2, '건의': 5},
    {'주차': '2주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 118, '싸지방': 58, 'PX': 138, '수면만족': 3.8, '식단만족': 3.8, '건의': 2},
    {'주차': '2주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 92, '싸지방': 29, 'PX': 166, '수면만족': 3.4, '식단만족': 3.4, '건의': 5},
    {'주차': '2주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 106, '싸지방': 50, 'PX': 121, '수면만족': 3.6, '식단만족': 3.1, '건의': 7},
    {'주차': '3주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 121, '싸지방': 61, 'PX': 142, '수면만족': 3.7, '식단만족': 4.0, '건의': 4},
    {'주차': '3주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 85, '싸지방': 26, 'PX': 171, '수면만족': 3.2, '식단만족': 3.4, '건의': 7},
    {'주차': '3주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 103, '싸지방': 49, 'PX': 117, '수면만족': 3.7, '식단만족': 3.3, '건의': 9},
    {'주차': '4주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 125, '싸지방': 66, 'PX': 135, '수면만족': 3.7, '식단만족': 4.1, '건의': 3},
    {'주차': '4주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 80, '싸지방': 24, 'PX': 178, '수면만족': 3.0, '식단만족': 3.3, '건의': 8},
    {'주차': '4주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 108, '싸지방': 52, 'PX': 120, '수면만족': 3.7, '식단만족': 3.2, '건의': 12}
]

df = pd.DataFrame(data_records)

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드합니다."""
    filename = f"k14_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K14", "../uploads/K14"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 분석 결과 및 검증 리포트를 시각적으로 렌더링합니다."""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 분석 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ [K14] 실습 환경, 12행 생활관 데이터셋(df) 및 시각화 렌더러 준비 완료!")
display(df)



### 📚 공통 가상 배경 및 데이터 모음

**K14 실습자료 - 군 생활 데이터 분석 - 생활관 생활 데이터로 개선 과제 찾기**

구    분 **실습자료 [교육용가상자료]**

**1. 사용 안내**

이 자료는 '군 생활 데이터 분석' 실습에 쓰는 교육용 가상 자료입니다. 가상 한빛대대의 이용 수·설문 점수·건의 건수는 모두 임의로 만든 수치로 실제 부대의 현황과 무관합니다. 모든 값은 중대 단위로 모은 집계 값이며 개인을 식별할 수 있는 정보가 없습니다. 생성형 AI에는 이 자료만 입력하고, 분석 결과는 참고용으로만 사용합니다.

**2. 가상 상황**

| **부대** | 가상 한빛대대 (교육용 가상 부대) |
| --- | --- |

| **상황** | 대대 복지 담당 병사인 학습자는 4주간의 생활관 생활 데이터를 보고, 중대장 회의에 낼 '생활관 개선 과제'를 정리하려 한다. |
| --- | --- |

| **원칙** | AI가 계산한 수치는 직접 확인한다. 원인은 가설로만 쓰고, 결과는 참고용이다. |
| --- | --- |

**3. 가상 생활관 생활 데이터 (4주 × 3개 중대)**

| **주차** | **중대** | **인원(명)** | **체력단련실 이용(회)** | **사이버지식정보방 이용(회)** | **PX 이용(회)** | **수면 만족(5점)** | **식단 만족(5점)** | **건의(건)** |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1주차 | 1중대 | 48 | 112 | 54 | 140 | 3.8 | 3.9 | 3 |
| 1주차 | 2중대 | 52 | 98 | 31 | 160 | 3.6 | 3.5 | 4 |
| 1주차 | 3중대 | 45 | 104 | 47 | 118 | 3.7 | 3.2 | 5 |
| 2주차 | 1중대 | 48 | 118 | 58 | 138 | 3.8 | 3.8 | 2 |
| 2주차 | 2중대 | 52 | 92 | 29 | 166 | 3.4 | 3.4 | 5 |
| 2주차 | 3중대 | 45 | 106 | 50 | 121 | 3.6 | 3.1 | 7 |
| 3주차 | 1중대 | 48 | 121 | 61 | 142 | 3.7 | 4.0 | 4 |
| 3주차 | 2중대 | 52 | 85 | 26 | 171 | 3.2 | 3.4 | 7 |
| 3주차 | 3중대 | 45 | 103 | 49 | 117 | 3.7 | 3.3 | 9 |
| 4주차 | 1중대 | 48 | 125 | 66 | 135 | 3.7 | 4.1 | 3 |
| 4주차 | 2중대 | 52 | 80 | 24 | 178 | 3.0 | 3.3 | 8 |
| 4주차 | 3중대 | 45 | 108 | 52 | 120 | 3.7 | 3.2 | 12 |

| 이용 수는 그 주에 중대 장병이 해당 시설을 이용한 총 횟수, 수면·식단 만족은 중대 설문 평균(1점 매우 불만 ~ 5점 매우 만족)입니다. |
| --- |

**4. 건의 유형별 건수 (4주 합계)**

| **중대** | **시설·환경** | **식단** | **일과·시간** | **기타** | **합계** |
| --- | --- | --- | --- | --- | --- |
| 1중대 | 4 | 3 | 3 | 2 | 12 |
| 2중대 | 6 | 4 | 10 | 4 | 24 |
| 3중대 | 17 | 9 | 3 | 4 | 33 |

**5. 특이사항 메모 (가설 세우기용)**

| **주차** | **대상** | **메모** |
| --- | --- | --- |
| 1주차 | 3중대 | 난방기 이상 소음 건의가 접수됨(점검 전) |
| 2주차 | 2중대 | 야간 경계 근무 교대 시간이 조정됨 |
| 3주차 | 전 중대 | 사격 훈련 준비로 개인정비 시간이 줄어듦 |
| 4주차 | 전 중대 | 동계 피복 보급 완료, 3중대 난방기 점검 예정 공지 |

| 메모는 원인을 확정하는 근거가 아니라 가설을 세울 때 참고하는 단서입니다. |
| --- |

**미션별 작성 서식**


---

## 🎯 미션 01. 생활관 데이터 항목 이해하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **생활관 생활 데이터의 항목과 의미를 이해합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **실습자료의 4주간 중대별 데이터 표에서 항목(체력단련실·사이버지식정보방·PX 이용 수, 수면 만족 점수, 식단 만족도, 건의 건수)의 단위와 뜻을 정리하고, 분석으로 알고 싶은 질문 3개를 정 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `데이터 항목 설명표, 분석 질문 3개, 개인 식별 정보 점검 결과` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 1 데이터 항목 설명표

| **항목** | **단위** | **뜻** | **분석에 쓸 수 있는 질문** |
| --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 문제 정의 — 데이터 항목을 이해하고 분석 질문과 개인정보 점검을 정리했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(데이터 항목 설명표, 분석 질문 3개, 개인 식별 정보 점검 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 01.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_01_prompt = """[생활관 데이터 6대 지표 단위/정의 및 분석 질문 도출 질의]
당신은 부대 생활 데이터 분석관입니다. 가상 한빛대대의 4주간 생활관 생활 데이터셋에 포함된 6대 핵심 지표(체력단련실, 사이버지식정보방, PX, 수면 만족도, 식단 만족도, 건의 건수)에 대해 측정 단위, 데이터 성격(총합/평균), 실무적 관리 목적을 표로 정립해 주십시오.
또한 본 12행 데이터셋을 통해 검증 가능한 3대 핵심 분석 질문을 도출하고, 개인식별정보(PII) 비포함 및 군사보안 준수 확인 체크리스트를 작성해 주십시오."""

mission_01_result = """## 1. 데이터 항목 설명표
| 항목명 | 측정 단위 | 데이터 성격 (총합 / 평균 등) | 실무적 관리 목적 |
| :--- | :--- | :--- | :--- |
| 체력단련실 이용 | 회 | 주간 중대 총 이용 횟수 | 장병 체력 증진 및 시설 수요 파악 |
| 사이버지식정보방 이용 | 회 | 주간 중대 총 이용 횟수 | 자기개발 및 외부 소통 여건 점검 |
| PX(복지매장) 이용 | 회 | 주간 중대 총 이용 횟수 | 복지 만족도 및 휴식 패턴 분석 |
| 수면 만족도 | 점수 (1~5점) | 중대 장병 주간 설문 평균 | 생활관 환경(소음, 온도) 및 피로도 진단 |
| 식단 만족도 | 점수 (1~5점) | 중대 장병 주간 설문 평균 | 급식 만족도 및 식생활 개선 요소 발굴 |
| 생활관 건의 | 건 | 주간 접수된 총 건의 건수 | 병영 애로사항 조기 식별 및 사고 예방 |

## 2. 분석을 통해 알고 싶은 핵심 질문 3개
1. **질문 1**: (예: 4주 동안 3개 중대 중 체력단련실과 싸지방 이용률이 지속적으로 감소하거나 증가하는 중대가 있는가?)
2. **질문 2**: (예: 인원수가 다른 중대들을 비교할 때 1인당 시설 이용량과 만족도 간에 유의미한 차이가 나타나는가?)
3. **질문 3**: (예: 건의 건수가 급증한 중대에서 특정 만족도 점수가 함께 하락하는 상관관계가 존재하는가?)

## 3. 개인 식별 정보(PII) 및 보안 점검 결과
- [x] 성명, 군번, 계급, 주민번호 등 개인 식별 요소가 일체 포함되어 있지 않음을 확인했습니다.
- [x] 본 데이터는 개별 장병의 기록이 아닌 '중대 단위 집계 수치'임을 확인했습니다.
- [x] 실제 부대 정보가 아닌 교육용 가상 데이터셋(가상 한빛대대)만을 활용합니다.

## 4. 도전 과제: 추가 분석 질문
- **추가 질문**: 
- **선정 사유 및 검증 방안**:"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 01. 생활관 데이터 항목 이해하기", mission_01_prompt, mission_01_result)



---

## 🎯 미션 02. 데이터 표 정리하고 점검하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **AI에 넣을 데이터를 표로 정리하고 이상한 값을 점검합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **12행(4주 × 3개 중대)의 표를 AI가 읽기 쉬운 형태로 정리하고, 빠진 값·단위 혼동·범위를 벗어난 값이 없는지 점검 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `정리한 데이터 표, 데이터 점검 기록` 완성

### 📑 실습 서식 및 대조 점검 양식
실습가이드 지침에 따라 가상 데이터를 분석하고 프롬프트/점검표를 완성합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 정보 구조화 — 데이터를 읽기 쉬운 표로 정리하고 이상한 값과 비교 기준을 점검했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(정리한 데이터 표, 데이터 점검 기록)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 02.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_02_prompt = """[12행 생활 데이터 마크다운 표 정형화 및 무결성 검증 질의]
당신은 군 데이터 정제 및 무결성 검증 전문가입니다. 가상 한빛대대 3개 중대의 4주간 생활관 생활 데이터 원문을 AI가 분석하기 용이한 12행 표준 마크다운 표로 정형화해 주십시오.
또한 결측치, 만족도 점수 범위(1~5점 척도), 단위 일관성을 점검하는 데이터 무결성 점검 기록을 작성하고, 중대별 인원 차이(1중대 48명, 2중대 52명, 3중대 45명)를 고려한 1인당 지표 산출 계획을 수립해 주십시오."""

mission_02_result = """## 1. 정형 마크다운 데이터 표 (12행)
| 주차 | 중대 | 인원(명) | 체력단련실(회) | 싸지방(회) | PX(회) | 수면만족(5점) | 식단만족(5점) | 건의(건) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 1주차 | 1중대 | 48 | 112 | 54 | 140 | 3.8 | 3.9 | 3 |
| 1주차 | 2중대 | 52 | 98 | 31 | 160 | 3.6 | 3.5 | 4 |
| 1주차 | 3중대 | 45 | 104 | 47 | 118 | 3.7 | 3.2 | 5 |
| 2주차 | 1중대 | 48 | 118 | 58 | 138 | 3.8 | 3.8 | 2 |
| 2주차 | 2중대 | 52 | 92 | 29 | 166 | 3.4 | 3.4 | 5 |
| 2주차 | 3중대 | 45 | 106 | 50 | 121 | 3.6 | 3.1 | 7 |
| 3주차 | 1중대 | 48 | 121 | 61 | 142 | 3.7 | 4.0 | 4 |
| 3주차 | 2중대 | 52 | 85 | 26 | 171 | 3.2 | 3.4 | 7 |
| 3주차 | 3중대 | 45 | 103 | 49 | 117 | 3.7 | 3.3 | 9 |
| 4주차 | 1중대 | 48 | 125 | 66 | 135 | 3.7 | 4.1 | 3 |
| 4주차 | 2중대 | 52 | 80 | 24 | 178 | 3.0 | 3.3 | 8 |
| 4주차 | 3중대 | 45 | 108 | 52 | 120 | 3.7 | 3.2 | 12 |

## 2. 데이터 무결성 점검 기록
- **결측치(빈 셀) 점검**: 108개 셀 중 결측치 ( 0 )건 확인
- **만족도 점수 범위(1.0~5.0) 점검**: 비정상 수치 ( 없음 )
- **단위 일관성 점검**: 이용량(회), 점수(5점 척도), 건의(건) 표기 일치 여부 확인 완료

## 3. 1인당 지표 산출 계획
- **분모 기준**: 1중대 ( 48 )명, 2중대 ( 52 )명, 3중대 ( 45 )명
- **산출 공식**: `1인당 4주 총 이용 횟수 = 4주간 총 이용 횟수 ÷ 중대 인원수`
- **적용 대상 항목**: 체력단련실, 싸지방, PX 이용 건수 (만족도 점수는 평균이므로 제외)

## 4. 도전 과제: 이상치 탐지 실험
- **변형 데이터 내용**: 
- **AI의 이상치 감지 여부 및 반응**:"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 02. 데이터 표 정리하고 점검하기", mission_02_prompt, mission_02_result)



In [ ]:
# [미션 02 실습 보조] 결측치 및 데이터 무결성 검증 (Pandas)
print('--- 결측치(Null) 점검 ---')
print(df.isnull().sum())

print('\n--- 만족도 점수 범위 점검 (1~5점 척도 여부) ---')
print(df[['수면만족', '식단만족']].describe().loc[['min', 'max']])

print('\n--- 1인당 계산용 중대별 정원 현황 ---')
display(df.groupby('중대')['인원(명)'].first().to_frame())

---

## 🎯 미션 03. 데이터 요약 요청하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **AI에게 데이터 요약을 요청합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **데이터 표를 붙이고 '항목별 4주 합계·평균과 가장 높거나 낮은 중대를 표로 정리하라'는 요약 프롬프트를 설계하고 작성하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `요약 프롬프트, AI 요약 결과` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 3 데이터 요약 프롬프트 틀

| 너는 가상 한빛대대의 생활 데이터를 쉽게 설명하는 데이터 분석 도우미다.
아래 [데이터]만 사용하고, 없는 값은 만들지 마라.
항목별로 4주 합계와 주간 평균, 가장 높은 중대와 낮은 중대를 표로 정리하라.
계산 과정을 한 줄로 함께 보여라.
[데이터]
(데이터 표 붙여 넣기) |
| --- |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 프롬프트 작성(분석 요청) — 데이터를 제공하고 요약 항목과 형식을 지정해 분석을 요청했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(요약 프롬프트, AI 요약 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 03.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_03_prompt = """당신은 군 생활 데이터 분석관입니다. 아래 제공하는 한빛대대 3개 중대의 4주간 생활관 생활 데이터(12행)를 분석하여 마크다운 표로 요약해 주세요.

[요구사항]
1. 6개 항목(체력단련실, 싸지방, PX, 수면만족, 식단만족, 건의)별 4주 합계와 주간 평균을 산출해 주세요.
2. 각 항목별 4주 합계(또는 평균) 기준 최고 중대와 최저 중대를 명시해 주세요.
3. 결과는 반드시 가독성이 높은 마크다운 표 형태로 출력해 주세요."""

mission_03_result = """## 1. 표준 요약 프롬프트 전문
```text
당신은 군 생활 데이터 분석관입니다. 아래 제공하는 한빛대대 3개 중대의 4주간 생활관 생활 데이터(12행)를 분석하여 마크다운 표로 요약해 주세요.

[요구사항]
1. 6개 항목(체력단련실, 싸지방, PX, 수면만족, 식단만족, 건의)별 4주 합계와 주간 평균을 산출해 주세요.
2. 각 항목별 4주 합계(또는 평균) 기준 최고 중대와 최저 중대를 명시해 주세요.
3. 결과는 반드시 가독성이 높은 마크다운 표 형태로 출력해 주세요.
```

## 2. AI 요약 결과 기록표
| 항목 | 1중대 (합계 / 주평균) | 2중대 (합계 / 주평균) | 3중대 (합계 / 주평균) | 최고 중대 | 최저 중대 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 체력단련실(회) | 476회 / 119.0회 | 355회 / 88.8회 | 421회 / 105.3회 | 1중대 | 2중대 |
| 싸지방(회) | 239회 / 59.8회 | 110회 / 27.5회 | 198회 / 49.5회 | 1중대 | 2중대 |
| PX(회) | 555회 / 138.8회 | 675회 / 168.8회 | 476회 / 119.0회 | 2중대 | 3중대 |
| 수면만족(5점) | - / 3.75점 | - / 3.30점 | - / 3.68점 | 1중대 | 2중대 |
| 식단만족(5점) | - / 3.95점 | - / 3.40점 | - / 3.20점 | 1중대 | 3중대 |
| 건의(건) | 12건 / 3.0건 | 24건 / 6.0건 | 33건 / 8.3건 | 3중대 | 1중대 |

## 3. 도전 과제: 요약 조건 변형 실험
- **적용한 변형 조건**: (예: 1주차 대비 4주차 증감률 추가)
- **변형 요약 결과 및 평가**:"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 03. 데이터 요약 요청하기", mission_03_prompt, mission_03_result)



---

## 🎯 미션 04. 요약 수치 직접 확인하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **AI가 계산한 수치를 직접 확인합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **AI가 알려 준 합계·평균·증감률을 계산기나 표 계산 도구로 다시 계산해 맞음·틀림으로 검토하고 핵심 정보를 식별하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `수치 검산표, 오류 원인과 수정 요청` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 4 수치 검산표

| **항목·중대** | **AI가 알려 준 값** | **직접 계산한 값** | **맞음·틀림** | **오류 원인** |
| --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] AI 결과 검증 — AI가 계산한 수치를 직접 다시 계산해 오류를 찾고 바로잡았는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(수치 검산표, 오류 원인과 수정 요청)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 04.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_04_prompt = """[AI 산출 수치 1:1 수기 검산 및 CoT 오류 수정 피드백 질의]
당신은 데이터 팩트체크 및 HITL(Human-in-the-Loop) 검증 전문가입니다. 앞서 AI가 산출한 4주간 합계 수치에 대해 원문 데이터를 바탕으로 1:1 수기 검산을 수행하고 판정표(PASS/FAIL)를 작성해 주십시오.
특히 계산 착오가 발생하기 쉬운 항목(예: 2중대 PX 이용 합계 665 오산 등)을 식별하여 오류 원인을 분석하고, 생각의 사슬(CoT, Chain-of-Thought) 기법을 적용하여 올바른 수치(675)를 도출하도록 단계별 덧셈 과정을 유도하는 피드백 프롬프트를 작성해 주십시오."""

mission_04_result = """## 1. 1:1 수기 검산 및 판정표
| 항목 및 대상 | 원문 데이터 (1~4주차) | AI 제시값 | 직접 검산값 | 판정 (PASS/FAIL) | 오류 원인 분석 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 1중대 체력 합계 | 112 + 118 + 121 + 125 | 476 | 476 | [x] PASS | 일치 |
| 2중대 체력 합계 | 98 + 92 + 85 + 80 | 355 | 355 | [x] PASS | 일치 |
| 3중대 체력 합계 | 104 + 106 + 103 + 108 | 421 | 421 | [x] PASS | 일치 |
| 1중대 싸지방 합계 | 54 + 58 + 61 + 66 | 239 | 239 | [x] PASS | 일치 |
| 2중대 싸지방 합계 | 31 + 29 + 26 + 24 | 110 | 110 | [x] PASS | 일치 |
| 3중대 싸지방 합계 | 47 + 50 + 49 + 52 | 198 | 198 | [x] PASS | 일치 |
| 2중대 PX 합계 | 160 + 166 + 171 + 178 | 665 (예시 오류) | 675 | [x] FAIL | 178을 168로 단순 오산 |
| 3중대 건의 합계 | 5 + 7 + 9 + 12 | 33 | 33 | [x] PASS | 일치 |

## 2. CoT 오류 수정 요청 프롬프트 및 교정본
- **피드백 프롬프트**:
  ```text
  2중대 4주간 PX 이용 합계에서 계산 착오가 발생했습니다. 
  1주차(160), 2주차(166), 3주차(171), 4주차(178) 수치를 하나씩 순서대로 더하는 과정(CoT)을 단계별로 보여주고 최종 올바른 합계를 도출해 주세요.
  ```
- **AI 교정 응답**:
  - 1단계: 160 + 166 = 326
  - 2단계: 326 + 171 = 497
  - 3단계: 497 + 178 = 675 (정정 완료)

## 3. 도전 과제: 재검산 일관성 비교
- **새 세션 재검산 결과**: 
- **HITL 검증의 중요성 메모**:"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 04. 요약 수치 직접 확인하기", mission_04_prompt, mission_04_result)



In [ ]:
# [미션 04 실습 보조] AI 산출 수치 직접 1:1 검산 (HITL 연산 검증)
print('=== 중대별 4주간 총합 (체련실/싸지방/PX/건의) ===')
totals = df.groupby('중대')[['체력단련실', '싸지방', 'PX', '건의']].sum()
display(totals)

print('\n=== 중대별 4주간 만족도 평균 (수면/식단) ===')
averages = df.groupby('중대')[['수면만족', '식단만족']].mean().round(2)
display(averages)

# AI 요약 결과와 대조하여 1:1 맞음/틀림을 판정하세요.

---

## 🎯 미션 05. 주차별 추세 파악하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **주차별 변화 추세를 파악합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **중대별로 1주차에서 4주차까지 늘었는지, 줄었는지, 비슷한지 AI에게 정리하게 하고 증감폭을 직접 확인하고 유효성을 점검하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `중대별 주차 추세표, 추세 해석 메모` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 5 중대별 주차 추세표

| **중대** | **항목** | **1주차** | **4주차** | **증감** | **추세처럼 보임 / 확정 어려움** |
| --- | --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 추세 분석 — 주차별 변화를 증감폭으로 정리하고 짧은 기간의 한계를 구분했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(중대별 주차 추세표, 추세 해석 메모)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 05.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_05_prompt = """[주차별 추세 분석 및 단정 방지 해석 질의]
당신은 시계열 데이터 분석관입니다. 3개 중대의 1주차 대비 4주차 생활관 생활 지표 변화를 비교하여 중대별 주차 추세표(증감폭, 증감률, 추세 방향)를 작성해 주십시오.
또한 단 4주간의 단기 수치만으로 성급하게 결론을 내리지 않고, '추세처럼 보임'과 '단정하기 어려움'을 명확히 구분하여 지휘관 보고 시 통계적 오판을 방지하기 위한 추세 해석 메모를 작성해 주십시오."""

mission_05_result = """## 1. 중대별 주차 추세표 (1주차 vs 4주차)
| 중대 | 항목 | 1주차 | 4주차 | 증감폭 (4주 - 1주) | 증감률 (%) | 추세 방향 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 1중대 | 체력단련실 | 112 | 125 | +13 | +11.6% | 지속 증가 |
| 2중대 | 체력단련실 | 98 | 80 | -18 | -18.4% | 지속 감소 |
| 3중대 | 체력단련실 | 104 | 108 | +4 | +3.8% | 유지/소폭 증가 |
| 2중대 | 수면 만족도 | 3.6 | 3.0 | -0.6 | -16.7% | 뚜렷한 하락 |
| 3중대 | 건의 건수 | 5 | 12 | +7 | +140.0% | 급격한 증가 |

## 2. 추세 해석 메모 (단정 금지 원칙 반영)
- **2중대 체력단련실 및 수면만족도**: 4주간 체력단련실 이용(-18회)과 수면 만족도(-0.6점)가 동반 하락하는 경향이 관찰됨. 단, 4주는 짧은 관찰 기간이므로 원인을 단정할 수 없으며, 2주차에 실시된 야간 경계 근무 교대 시간 조정과의 연계성을 추가 검토할 필요가 있음.
- **3중대 건의 건수 급증**: 1주차 5건에서 4주차 12건으로 2배 이상 증가함. 불만의 전반적 확산으로 확정하기 전에 어떤 유형의 건의가 집중되었는지 세부 데이터 대조가 요구됨.

## 3. 도전 과제: 2~3주차 중간 변곡점 분석
- **식별된 중간 변곡점**: (예: 3주차 전 중대 식단/수면 만족도 및 시설 이용 일시적 변화)
- **배경 원인 단서**: 3주차 사격 훈련 준비로 인한 개인정비 시간 축소 메모와 부합함."""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 05. 주차별 추세 파악하기", mission_05_prompt, mission_05_result)



---

## 🎯 미션 06. 중대별 비교하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **중대별 차이를 공정하게 비교합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **인원이 다른 3개 중대를 총합이 아니라 1인당 값으로 비교하게 하고, 총합과 1인당 값의 순위가 달라지는 항목이 있는지 확인하고 유효성을 점검하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `중대별 1인당 비교표, 주목할 항목 목록` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 6 1인당 비교표

| **항목** | **1중대(1인당)** | **2중대(1인당)** | **3중대(1인당)** | **총합 순위와 달라졌는가** |
| --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 비교 분석 — 인원 차이를 고려한 기준으로 중대별 차이를 비교하고 주목 항목을 골랐는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(중대별 1인당 비교표, 주목할 항목 목록)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 06.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_06_prompt = """[인원 차이 보정 1인당 정규화 비교 및 순위 역전 분석 질의]
당신은 부대 통계 정규화 분석 전문가입니다. 3개 중대의 인원수(1중대 48명, 2중대 52명, 3중대 45명)가 서로 다르므로 단순 총합 비교 시 발생하는 통계적 왜곡을 방지하기 위해, 4주 총 이용량을 중대 인원수로 나눈 '1인당 4주 총 이용량 비교표'를 산출해 주십시오.
총합 기준 순위와 1인당 정규화 순위 간에 발생한 순위 역전 현상(예: 체력단련실 2중대 vs 3중대)을 분석하고, 부대 관찰이 시급한 중점 관리 대상 지표 2개를 선정해 주십시오."""

mission_06_result = """## 1. 1인당 4주 총 이용량 비교표
| 항목 | 1중대 (48명) | 2중대 (52명) | 3중대 (45명) | 최고 중대 (1인당) |
| :--- | :--- | :--- | :--- | :--- |
| 체력단련실 1인당 | 9.92회 (476/48) | 6.83회 (355/52) | 9.36회 (421/45) | 1중대 |
| 싸지방 1인당 | 4.98회 (239/48) | 2.12회 (110/52) | 4.40회 (198/45) | 1중대 |
| PX 1인당 | 11.56회 (555/48) | 12.98회 (675/52) | 10.58회 (476/45) | 2중대 |

## 2. 총합 순위 vs 1인당 순위 역전 분석표
| 항목 | 단순 총합 순위 | 1인당 평균 순위 | 역전 여부 및 원인 |
| :--- | :--- | :--- | :--- |
| 체력단련실 | 1위: 1중대(476), 2위: 3중대(421), 3위: 2중대(355) | 1위: 1중대(9.92), 2위: 3중대(9.36), 3위: 2중대(6.83) | 순위 유지 |
| 싸지방 | 1위: 1중대(239), 2위: 3중대(198), 3위: 2중대(110) | 1위: 1중대(4.98), 2위: 3중대(4.40), 3위: 2중대(2.12) | 순위 유지 (2중대 격차 극대화 확인) |
| PX 이용 | 1위: 2중대(675), 2위: 1중대(555), 3위: 3중대(476) | 1위: 2중대(12.98), 2위: 1중대(11.56), 3위: 3중대(10.58) | 순위 유지 (2중대의 압도적 PX 의존도 확인) |

## 3. 주목할 핵심 항목 2~3개 및 선정 사유
1. **2중대의 저조한 시설 이용률**: 체력단련실(6.8회), 싸지방(2.1회) 모두 타 중대 대비 1인당 이용량이 현저히 낮음.
2. **3중대의 건의 집중 현상**: 인원수는 가장 적으나(45명) 건의 건수는 33건으로 대대 전체의 47.8%를 차지함.

## 4. 도전 과제: 100명당 건의 발생 비율
- **1중대**: 25.0건 / 100명
- **2중대**: 46.2건 / 100명
- **3중대**: 73.3건 / 100명 (타 중대 대비 3배 가까운 체감 건의 강도)"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 06. 중대별 비교하기", mission_06_prompt, mission_06_result)



In [ ]:
# [미션 06 실습 보조] 중대별 1인당 4주 총 이용 횟수 계산 (정규화)
personnel = df.groupby('중대')['인원(명)'].first()
totals = df.groupby('중대')[['체력단련실', '싸지방', 'PX', '건의']].sum()
per_capita = totals.div(personnel, axis=0).round(2)

print('=== 1인당 4주 총 이용 횟수 (총합 ÷ 인원수) ===')
display(per_capita[['체력단련실', '싸지방', 'PX']])

print('\n=== 장병 100명당 건의 발생률 ===')
suggestions_rate = (totals['건의'].div(personnel) * 100).round(1).to_frame(name='100명당 건의 건수')
display(suggestions_rate)

---

## 🎯 미션 07. 그래프 시각화 요청하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **그래프 시각화를 요청하고 오해를 막습니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **추세에는 선 그래프, 중대 비교에는 막대 그래프처럼 목적에 맞는 그래프를 AI에게 요청 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `그래프 요청 프롬프트 2개, 그래프 점검표` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 7 그래프 요청 프롬프트 틀과 점검표

| 위 [데이터]로 (선 그래프 / 막대 그래프)를 만들어 줘.
- 그릴 항목: 
- 가로축: 
- 세로축 이름과 단위: 
- 세로축은 0부터 시작하게 하거나 시작값을 명시해 줘.
- 제목과 범례를 넣어 줘.
그래프를 그릴 수 없으면 표 계산 도구에서 만드는 방법을 단계별로 알려 줘. |
| --- |

| **점검 항목** | **그래프 1** | **그래프 2** |
| --- | --- | --- |
| 축 이름과 단위가 맞다 | ( ) | ( ) |
| 축 시작값이 과장되지 않았다 | ( ) | ( ) |
| 값이 표와 일치한다 | ( ) | ( ) |
| 제목이 내용을 설명한다 | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 시각화 요청·검증 — 목적에 맞는 그래프를 요청하고 축과 단위를 데이터와 대조해 점검했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(그래프 요청 프롬프트 2개, 그래프 점검표)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 07.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_07_prompt = """[목적별 맞춤형 시각화 프롬프트 2종 및 왜곡 방지 점검 질의]
당신은 군 데이터 시각화 전문가입니다. 한빛대대 생활 데이터 분석 결과를 지휘관 주관 회의에서 효과적으로 전달할 수 있도록 목적에 맞는 그래프 요청 프롬프트 2종을 작성해 주십시오:
1. 추세용: 3개 중대의 1~4주차 수면 만족도 변화를 나타내는 선 그래프(Line Chart)
2. 비교용: 3개 중대의 1인당 체력단련실 총 이용량을 비교하는 막대 그래프(Bar Chart)
그래프 생성 후 축 이름, 측정 단위, Y축 시작값(0점 척도 과장 여부)을 점검하는 시각화 품질 점검표를 함께 작성해 주십시오."""

mission_07_result = """## 1. 목적별 그래프 생성 프롬프트 2종
- **프롬프트 1 (추세용 - 선 그래프)**:
  ```text
  한빛대대 3개 중대의 1~4주차 수면 만족도 변화를 선 그래프(Line Chart)로 시각화해 줘.
  X축: 주차 (1주차~4주차), Y축: 수면 만족도 점수 (0점~5점 전체 표시)
  중대별 선 색상과 마커를 다르게 지정하고 범례를 명시해 줘.
  ```
- **프롬프트 2 (비교용 - 막대 그래프)**:
  ```text
  3개 중대의 '1인당 4주 체력단련실 이용 횟수'를 비교하는 막대 그래프(Bar Chart)를 생성해 줘.
  Y축 시작점은 반드시 0부터 시작하도록 설정하고(ylim 0부터), 축 라벨과 단위를 명확히 적어 줘.
  ```

## 2. 그래프 왜곡 점검표 (Checklist)
| 점검 항목 | 점검 기준 | 충족 여부 | 조치 내용 |
| :--- | :--- | :--- | :--- |
| 1. 축 라벨 및 단위 | X/Y축에 명칭과 단위(회, 점)가 표시되었는가? | [x] PASS | 주차, 점수 단위 표기 완료 |
| 2. Y축 시작값 (하한선) | 막대 그래프의 Y축이 0부터 시작하는가? | [x] PASS | 0점 기준 확인 (왜곡 방지) |
| 3. 범례 및 색상 | 중대별 구분이 뚜렷하고 데이터가 일치하는가? | [x] PASS | 1중대(파랑), 2중대(주황), 3중대(초록) |
| 4. 데이터 포인트 일치 | 차트 상의 수치가 검산 결과와 일치하는가? | [x] PASS | 1:1 수치 매핑 확인 |

## 3. 축 왜곡 교정 기록
- **발견된 왜곡**: 만족도 그래프에서 Y축 범위를 3.0~4.0으로 잡을 경우 0.2점 차이가 극단적으로 과장됨.
- **교정 조치**: Y축 범위를 0~5점 척도 전체로 설정하여 객관적인 격차 체감을 유도함.

## 4. 도전 과제: 축 시작값 변형 시각 비교 메모
- **관찰 결과**:"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 07. 그래프 시각화 요청하기", mission_07_prompt, mission_07_result)



In [ ]:
# [미션 07 실습 보조] AI 생성 시각화 코드 실행 및 왜곡(축/스케일) 점검
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. 추세 선 그래프 (수면 만족도)
for comp in ['1중대', '2중대', '3중대']:
    sub = df[df['중대'] == comp]
    axes[0].plot(sub['주차'], sub['수면만족'], marker='o', linewidth=2, label=comp)
axes[0].set_title('주차별 수면 만족도 변화 추이 (Line Chart)', fontsize=13)
axes[0].set_ylim(0, 5)  # 5점 척도 전체 표시 (왜곡 방지)
axes[0].set_ylabel('만족도 점수 (1~5점)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

# 2. 비교 막대 그래프 (1인당 체력단련실 이용)
comp_names = ['1중대', '2중대', '3중대']
personnel = df.groupby('중대')['인원(명)'].first()
gym_per_capita = df.groupby('중대')['체력단련실'].sum().div(personnel).loc[comp_names]
bars = axes[1].bar(comp_names, gym_per_capita, color=['#3366cc', '#dc3912', '#109618'], width=0.5)
axes[1].set_title('중대별 1인당 4주 체력단련실 이용 횟수 (Bar Chart)', fontsize=13)
axes[1].set_ylim(0, 12)  # Y축 0부터 시작 (정직한 시각화)
axes[1].set_ylabel('1인당 평균 이용 횟수(회)')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

# 막대 상단 수치 표기
for bar in bars:
    h = bar.get_height()
    axes[1].annotate(f'{h:.2f}회', xy=(bar.get_x() + bar.get_width()/2, h),
                     xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---

## 🎯 미션 08. 원인 가설과 개선 제안 만들기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **원인 가설과 개선 제안을 만듭니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **주목할 항목마다 데이터와 특이사항 메모를 근거로 원인 가설을 AI와 함께 2~3개 세우고, 각 가설을 확인하려면 어떤 추가 자료가 필요한지 적습니다 과정을 진행하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `원인 가설 표, 추가 확인 자료, 개선 제안 우선순위` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 8 원인 가설 표

| **주목 항목** | **가설** | **근거(데이터·메모)** | **추가로 확인할 자료** | **개선 제안** | **우선순위** |
| --- | --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 가설·제안 도출 — 근거 있는 가설과 확인 방법, 현실적인 개선 제안을 정리했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(원인 가설 표, 추가 확인 자료, 개선 제안 우선순위)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 08.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_08_prompt = """[다각도 원인 가설 수립 및 개선 제안 우선순위 도출 질의]
당신은 병영생활 개선 정책 연구관입니다. 앞서 식별한 이상 징후(3중대 건의 급증 33건, 2중대 수면 만족도 급감 및 체련실 이용 저조)에 대해, 가상 데이터셋의 '건의 유형별 통계'와 '주차별 특이사항 메모'를 연계하여 다각도의 원인 가설(2~3개)을 수립해 주십시오.
각 가설을 지지하는 정황 증거와 단정 방지 문구를 포함하고, 가설 검증을 위해 추가로 확인해야 할 부대 자료 목록 및 긴급성/실현성을 고려한 개선 제안 우선순위표를 도출해 주십시오."""

mission_08_result = """## 1. 원인 가설 수립표
| 주목 지표 현상 | 다각도 원인 가설 (2~3개) | 가설 지지 정황 단서 (특이사항 메모) |
| :--- | :--- | :--- |
| **3중대 건의 급증 (4주 33건)** | 1. 생활관 난방기 이상 소음으로 인한 수면 방해 가능성<br>2. 3중대 건의함 접근성 개선 또는 분대장 적극 수렴<br>3. 환절기 온습도 불균형에 따른 시설 불만 | - 1주차: 3중대 난방기 이상 소음 접수(점검 전)<br>- 4주차: 3중대 난방기 점검 예정 공지<br>- 건의 유형 중 '시설·환경'이 17건으로 과반 |
| **2중대 체력단련실 이용 급감** | 1. 2주차 야간 경계 근무 교대 시간 조정으로 인한 피로 누적<br>2. 특정 운동 기구 고장 또는 정비 소요 발생 가능성 | - 2주차 메모: 야간 근무 교대 시간 조정됨<br>- 건의 유형 중 '일과·시간' 건의가 10건으로 최다 |

## 2. 추가 확인 자료 목록
1. **대대 시설관리반 정비 일지**: 3중대 난방기 실제 수리 일자 및 부품 교체 내역 확인
2. **2중대 일과운영표 및 근무 편성표**: 야간 경계근무 변경 전후의 수면 및 체력단련 가능 시간대 비교

## 3. 개선 제안 우선순위표
| 우선순위 | 구분 | 개선 제안 과제 | 예상 소요 및 기대 효과 |
| :--- | :--- | :--- | :--- |
| **1순위 (긴급)** | 시설 정비 | 3중대 난방기 점검 조기 완료 및 소음 방지 조치 | 즉각적인 건의 감소 및 수면 환경 안정화 |
| **2순위 (단기)** | 일과 조정 | 2중대 야간근무자 전용 체력단련 및 휴식 시간대 지정 | 체력단련실 이용 회복 및 피로도 개선 |
| **3순위 (중장기)** | 제도 개선 | 중대별 건의함 온라인화 및 주간 처리 결과 게시판 공지 | 장병 소통 만족도 제고 및 신뢰 증진 |

*안내: 본 개선 제안은 가상 데이터 분석에 근거한 건의안이며, 최종 실행 여부는 부대 지휘계통 결심에 따름.*

## 4. 도전 과제: 대안 가설 및 확증 편향 점검
- **대안 가설**: 
- **비판적 검토 결과**:"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 08. 원인 가설과 개선 제안 만들기", mission_08_prompt, mission_08_result)



---

## 🎯 미션 09. 발표 자료 구성하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **발표 자료의 구성을 만듭니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **결론 먼저 말하는 5장 구성(핵심 요약, 현황, 원인 가설, 개선 제안, 확인할 점)을 AI와 함께 잡고 장마다 한 줄 핵심 문장과 사용할 그래프를 정 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `발표 자료 5장 구성안` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 9 발표 자료 5장 구성

| **장** | **제목** | **한 줄 핵심 문장** | **쓸 그래프** |
| --- | --- | --- | --- |
| 1 | 핵심 요약 | ( ) | ( ) |
| 2 | 현황 | ( ) | ( ) |
| 3 | 원인 가설 | ( ) | ( ) |
| 4 | 개선 제안 | ( ) | ( ) |
| 5 | 확인할 점 | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 대상 맞춤 활용 — 청중에게 맞는 순서와 쉬운 표현으로 발표 자료를 구성했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(발표 자료 5장 구성안)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 09.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_09_prompt = """[지휘관 회의 보고용 결론 우선(두괄식) 5장 슬라이드 기획 질의]
당신은 군 지휘관 주관 참모회의 보고 자료 기획관입니다. 4주간의 생활관 데이터 분석 결과를 바쁜 지휘관에게 명확하고 간결하게 전달할 수 있도록 '결론 우선(두괄식) 5장 슬라이드 기획안'을 작성해 주십시오.
- 1장: 핵심 요약 (진단 결과 및 긴급 개선 과제 2건)
- 2장: 생활관 현황 데이터 (시설 이용 패턴 및 수면/식단 만족도)
- 3장: 원인 가설 및 정황 분석 (3중대 난방기 소음, 2중대 야간근무 조정)
- 4장: 개선 제안 우선순위 (즉각 실행 및 중장기 과제)
- 5장: 확인할 점 및 건의사항 (결심 요청사항)
슬라이드별 1줄 핵심 헤드라인과 매칭 시각자료를 함께 명시해 주십시오."""

mission_09_result = """## 1. 결론 우선 5장 슬라이드 기획안
| 슬라이드 | 구성 테마 | 핵심 내용 | 매칭 시각 자료 |
| :--- | :--- | :--- | :--- |
| **1장** | 핵심 요약 (Executive Summary) | 4주간 생활 데이터 진단 결과 및 긴급 개선 과제 2건 제안 | 핵심 지표 요약 인포그래픽 |
| **2장** | 생활관 현황 데이터 | 3개 중대 시설 이용 패턴 및 수면·식단 만족도 현황 | 1인당 시설 이용 비교 막대 차트 |
| **3장** | 원인 가설 및 정황 분석 | 3중대 건의 급증(난방기) 및 2중대 이용 감소(근무시간) 분석 | 주차별 수면만족/건의 추세 선 그래프 |
| **4장** | 개선 제안 우선순위 | 난방기 긴급 조치(1순위), 근무자 휴식 보장(2순위) 실행안 | 우선순위별 조치 로드맵 표 |
| **5장** | 확인할 점 및 건의사항 | 시설관리반 정비 확인 및 중대장 결심 요청사항 | 추가 확인 문서 목록 및 결재란 |

## 2. 슬라이드별 한 줄 핵심 헤드라인
- **1장**: "생활관 데이터 분석 결과, 3중대 시설 건의 긴급 정비와 2중대 근무자 휴식 여건 보장이 시급합니다."
- **2장**: "중대 간 1인당 체력단련실 이용은 최대 1.45배 격차를 보이며, 2중대의 시설 이용률이 가장 저조합니다."
- **3장**: "3중대 시설 건의는 난방기 소음 문제와 직결되어 있으며, 2중대는 야간근무 조정 이후 피로가 누적되었습니다."
- **4장**: "예산 투입 없는 즉각적 일과 조정과 조기 정비 점검으로 2주 내 생활 만족도 반등이 가능합니다."
- **5장**: "시설반 정비 일정 확정과 2중대 일과 탄력 운영에 대한 지휘관님의 결심을 건의드립니다."

## 3. 현장 중심 쉬운 말 발표 스크립트 초안
- "중대장님, 1인당 9.9회라는 수치는 장병들이 주 2~3회꼴로 체력단련실을 꾸준히 찾는다는 의미입니다..."

## 4. 도전 과제: 3분 압축 브리핑 메모
- **3분 발표 선별 슬라이드**: 1장(핵심요약) → 3장(원인) → 4장(개선안)"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 09. 발표 자료 구성하기", mission_09_prompt, mission_09_result)



---

## 🎯 미션 10. 생활관 개선 과제 보고서 완성하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **생활관 개선 과제 보고서를 완성합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **데이터 요약, 검산 결과, 추세·비교, 그래프, 가설과 개선 제안, 발표 구성을 한 편의 보고서로 묶습니다 과정을 진행하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `생활관 개선 과제 보고서(최종본)` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 10 보고서 목차

| 1. 분석 목적과 데이터 설명
2. 요약과 검산 결과
3. 추세와 중대별 비교
4. 그래프
5. 원인 가설과 개선 제안
6. 분석의 한계 (기간 4주, 집계 값, 가설)
7. 안내 문구: 가상 데이터 분석 결과이며 참고용, 실제 결정은 지휘계통 |
| --- |

**채점표**

| **미션** | **평가 항목** | **구분** | **판정** |
| --- | --- | --- | --- |
| 1 | 생활관 데이터 항목 이해하기 | 필수 | PASS / FAIL |
| 2 | 데이터 표 정리하고 점검하기 | 필수 | PASS / FAIL |
| 3 | 데이터 요약 요청하기 | 필수 | PASS / FAIL |
| 4 | 요약 수치 직접 확인하기 | 필수 | PASS / FAIL |
| 5 | 주차별 추세 파악하기 | 필수 | PASS / FAIL |
| 6 | 중대별 비교하기 | 선택 | PASS / FAIL |
| 7 | 그래프 시각화 요청하기 | 필수 | PASS / FAIL |
| 8 | 원인 가설과 개선 제안 만들기 | 필수 | PASS / FAIL |
| 9 | 발표 자료 구성하기 | 선택 | PASS / FAIL |
| 10 | 생활관 개선 과제 보고서 완성하기 | 필수 | PASS / FAIL |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 결과물 완성 — 분석 과정과 개선 제안을 정리하고 한계와 확인 안내를 포함해 완성했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(생활관 개선 과제 보고서(최종본))의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 10.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_10_prompt = """[생활관 개선 과제 종합 보고서 최종본 집대성 질의]
당신은 대대 복지 및 행정 실무 장병입니다. 미션 1부터 9까지 수행한 데이터 분석, HITL 검산, 정규화 비교, 시각화, 원인 가설 및 개선 제안을 하나로 집대성하여 지휘계통에 정식 보고할 [가상 한빛대대 생활관 생활 데이터 분석을 통한 병영생활 개선 과제 종합 보고서] 최종본을 작성해 주십시오.
보고서에는 Ⅰ. 분석 개요, Ⅱ. 주요 데이터 분석 결과, Ⅲ. 핵심 원인 가설 및 정황 증거, Ⅳ. 생활관 개선 제안 과제, 분석의 한계 및 유의사항, 교육용 가상 데이터 및 군 AI 거버넌스(HITL 지휘계통 결심 원칙) 고지문이 완벽히 포함되어야 합니다."""

mission_10_result = """## 1. 생활관 생활 개선 과제 종합 보고서
### [보고서 제목]: 가상 한빛대대 생활관 생활 데이터 분석을 통한 병영생활 개선 과제 보고서

### Ⅰ. 분석 개요
1. 목적: 4주간 생활관 생활 데이터 분석을 통한 부대 관리 현안 식별 및 맞춤형 복지 개선책 도출
2. 대상: 한빛대대 1중대(48명), 2중대(52명), 3중대(45명) 총 145명
3. 분석 기간: 2026년 10월 1주차 ~ 4주차 (총 4주간)

### Ⅱ. 주요 데이터 분석 결과
1. **시설 이용 현황**: 1중대는 체력단련(1인당 9.9회) 및 싸지방(4.9회) 이용이 가장 활발한 반면, 2중대는 체력단련(6.8회), 싸지방(2.1회) 이용이 크게 저조함.
2. **수면 및 식단 만족도**: 전 중대 식단 만족도는 3.2~4.1점으로 점진적 상승세이나, 2중대 수면 만족도는 3.6점에서 3.0점으로 급감.
3. **건의 건수 분석**: 3중대 건의 건수가 33건(전체의 47.8%)으로 집중되었으며, 세부적으로 시설·환경 건의가 17건으로 최다.

### Ⅲ. 핵심 원인 가설 및 정황 증거
- 3중대 난방기 이상 소음 접수 단서와 시설 건의 집중 현상 간의 높은 상관성 확인
- 2중대 야간 경계 근무 교대 시간 조정 이후 수면 만족도 및 체련실 이용 동반 하락 포착

### Ⅳ. 생활관 개선 제안 과제
- [1순위] 3중대 생활관 난방기 정밀 점검 및 방음/부품 교체 조기 완료
- [2순위] 2중대 야간근무자 대상 체력단련 보장 시간제 탄력 운영
- [3순위] 주간 단위 건의사항 처리 현황 게시판 공유제 도입

---

## 2. 분석의 한계 및 유의사항
- 본 분석은 4주간의 단기 집계 데이터에 기초하므로 계절적 요인 및 장기 추세를 완전히 대변하지 못할 수 있습니다.
- 설문 결과는 중대 평균 점수이므로 개별 장병의 특수한 고충을 모두 포착하는 데에는 한계가 있습니다.

## 3. 군 AI 거버넌스 및 확인 안내 문구
> **확인 안내**: 본 보고서의 데이터는 교육용 가상 데이터셋에 근거하여 생성형 AI의 지원을 받아 작성된 실습 결과물이며 실제 부대 현황과 무관합니다. 보고서에 제시된 분석 결과 및 개선 제안은 지휘관의 의사결정을 지원하는 참고 자료이며, 실제 부대 내 업무 적용 및 세부 조치 결정은 정식 지휘계통의 승인을 받아야 합니다.

작성자: 가상 한빛대대 복지담당 장병 (서명)
확인관: 중대장 / 행정보급관 (인)

## 4. 도전 과제: 차기 4주 후속 모니터링 계획서
- **점검 지표 1**: 3중대 난방기 수리 후 주간 건의 건수 (목표: 주 3건 이하)
- **점검 지표 2**: 2중대 일과 조정 후 수면 만족도 (목표: 3.5점 이상 회복)"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 10. 생활관 개선 과제 보고서 완성하기", mission_10_prompt, mission_10_result)

